In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. 파일 경로 지정 (방금 전 완성한 3대 가중치 통합 데이터)
input_file = r'F:\Download\Step5_119_Final_Integrated_Score.xlsx'

# 분할된 데이터 저장 경로 (모두 Step5 유지)
train_file = r'F:\Download\Step5_Train_Data.xlsx'
val_file = r'F:\Download\Step5_Val_Data.xlsx'
test_file = r'F:\Download\Step5_Test_Data.xlsx'

try:
    print("⏳ 1. 통합 스코어 데이터를 불러오는 중입니다...")
    df = pd.read_excel(input_file, engine='openpyxl')
    
    # 2. 데이터 전처리
    print("⚙️ 2. 모델 학습을 위해 데이터를 정제하고 변환합니다...")
    # 분석에 필요한 핵심 컬럼의 결측치 제거
    df = df.dropna(subset=['연령대', 'Gender', '최종_Total_수요점수'])
    
    # Label Encoding (문자열인 '연령대'와 '성별'을 AI가 이해할 수 있는 숫자로 변환)
    le_age = LabelEncoder()
    le_gender = LabelEncoder()
    df['연령대_코드'] = le_age.fit_transform(df['연령대'])
    df['성별_코드'] = le_gender.fit_transform(df['Gender'])
    
    # 인코딩 결과 매핑 가이드 (나중에 모델 규칙을 해석할 때 필요합니다)
    print("\n💡 [라벨 인코딩 변환 기준]")
    print(" - 성별 코드:", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))
    print(" - 연령대 코드:", dict(zip(le_age.classes_, le_age.transform(le_age.classes_))))
    
    # AI 모델에 학습시킬 3가지 핵심 컬럼만 추출 (X: 연령/성별, y: 통합 스코어)
    analysis_df = df[['연령대_코드', '성별_코드', '최종_Total_수요점수']]
    
    # 3. 데이터 분할 (Train 80% / Validation 10% / Test 10%)
    print("\n✂️ 3. 데이터를 8:1:1 비율로 정밀하게 분할합니다...")
    
    # 1차 분할: 임시 데이터 90% (Train+Val) / Test 데이터 10%
    df_temp, df_test = train_test_split(analysis_df, test_size=0.1, random_state=42)
    
    # 2차 분할: 임시 데이터 90% 중 1/9(전체의 10%)를 Val로, 나머지 8/9(전체의 80%)를 Train으로 설정
    df_train, df_val = train_test_split(df_temp, test_size=(1/9), random_state=42)
    
    print(f" 📊 분할 완료 데이터 건수: \n  - Train (학습용 80%): {len(df_train)}건 \n  - Validation (검증용 10%): {len(df_val)}건 \n  - Test (테스트용 10%): {len(df_test)}건")
    
    # 4. 분할된 데이터를 각각 저장
    print("\n💾 4. 분할된 데이터를 파일로 저장합니다...")
    df_train.to_excel(train_file, index=False, engine='openpyxl')
    df_val.to_excel(val_file, index=False, engine='openpyxl')
    df_test.to_excel(test_file, index=False, engine='openpyxl')
    
    print(f"🎉 성공! 세 가지 모델링용 데이터 파일이 모두 저장되었습니다.")
    
except FileNotFoundError:
    print(f"❌ '{input_file}' 파일을 찾을 수 없습니다. 이전 단계에서 데이터가 제대로 저장되었는지 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 1. 통합 스코어 데이터를 불러오는 중입니다...
⚙️ 2. 모델 학습을 위해 데이터를 정제하고 변환합니다...

💡 [라벨 인코딩 변환 기준]
 - 성별 코드: {'F': np.int64(0), 'M': np.int64(1)}
 - 연령대 코드: {'10대': np.int64(0), '20-29세': np.int64(1), '30-39세': np.int64(2), '40-49세': np.int64(3), '50-59세': np.int64(4), '60-69세': np.int64(5)}

✂️ 3. 데이터를 8:1:1 비율로 정밀하게 분할합니다...
 📊 분할 완료 데이터 건수: 
  - Train (학습용 80%): 57928건 
  - Validation (검증용 10%): 7241건 
  - Test (테스트용 10%): 7242건

💾 4. 분할된 데이터를 파일로 저장합니다...
🎉 성공! 세 가지 모델링용 데이터 파일이 모두 저장되었습니다.


In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import warnings

# 불필요한 sklearn 경고 메시지 숨기기
warnings.filterwarnings('ignore', category=UserWarning)

# ==========================================
# 1. 파일 경로 지정
# ==========================================
file_train = r'F:\Download\Step5_Train_Data.xlsx'
file_full_data = r'F:\Download\Step5_119_Final_Integrated_Score.xlsx'
file_demand_score = r'F:\Download\Step5_Base_Demand_Score_List.xlsx'
file_catalog = r'F:\Download\Step3_통합_보험데이터_최종_1개월_월환산완료.csv'

output_file = r'F:\Download\Step5_Hybrid_Recommendation_Result.xlsx'

try:
    print("⏳ 1. 추천 엔진 초기화 중...")
    df_train = pd.read_excel(file_train, engine='openpyxl')
    df_full = pd.read_excel(file_full_data, engine='openpyxl')
    df_score = pd.read_excel(file_demand_score, engine='openpyxl')
    
    try:
        df_catalog = pd.read_csv(file_catalog)
    except UnicodeDecodeError:
        df_catalog = pd.read_csv(file_catalog, encoding='cp949')

    # ==========================================
    # 2. ML 예측 엔진 학습
    # ==========================================
    le_age = LabelEncoder()
    le_gender = LabelEncoder()
    
    df_full['연령대'] = df_full['연령대'].fillna('알수없음').astype(str)
    df_full['Gender'] = df_full['Gender'].fillna('알수없음').astype(str)
    
    le_age.fit(df_full['연령대'])
    le_gender.fit(df_full['Gender'])
    
    X_train = df_train[['연령대_코드', '성별_코드']]
    y_train = df_train['최종_Total_수요점수']
    
    rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    rf_model.fit(X_train, y_train)

    # ==========================================
    # 3. 상품 가치 스코어링 & 회사명 추출
    # ==========================================
    print("📊 2. 보험 상품별 가치 계산 및 보험사 정보 추출 중...")
    score_dict = dict(zip(df_score['보험보장_항목'], df_score['Demand_Score(100)']))
    basic_cols = ['국가', '상품플랜', '가입나이', '성별', '최종보험료']
    coverage_cols = [col for col in df_catalog.columns if col not in basic_cols]
    
    def calculate_plan_benefit(row):
        total_benefit = 0
        for col in coverage_cols:
            amt = pd.to_numeric(row[col], errors='coerce')
            if pd.notna(amt) and amt > 0:
                weight = score_dict.get(col, 1.0)
                total_benefit += np.log1p(amt) * weight
        return total_benefit

    df_catalog['보장_충실도_점수'] = df_catalog.apply(calculate_plan_benefit, axis=1)
    df_catalog['최종보험료_num'] = pd.to_numeric(df_catalog['최종보험료'], errors='coerce').fillna(999999)
    df_catalog['가성비_점수'] = (df_catalog['보장_충실도_점수'] / (df_catalog['최종보험료_num'] + 1)) * 10000
    
    # 💡 [요청사항 반영] '상품플랜'에서 '보험사' 이름 추출 (보통 '_' 나 띄어쓰기 앞부분이 회사명임)
    df_catalog['추천_회사명'] = df_catalog['상품플랜'].apply(
        lambda x: str(x).split('_')[0] if '_' in str(x) else str(x).split(' ')[0]
    )

    # ==========================================
    # 스마트 입력값 매칭 함수
    # ==========================================
    def get_safe_label(encoder, target_val, fallback_idx=0):
        target_str = str(target_val).strip()
        classes = encoder.classes_
        if target_str in classes:
            return target_str
        for cls in classes:
            if target_str[:1] in str(cls) or target_str[:2] in str(cls):
                return cls
        return classes[fallback_idx]

    # ==========================================
    # 4. 하이브리드 추천 함수
    # ==========================================
    def recommend_insurance(target_age, target_gender):
        print(f"\n👤 [입력 프로필] 연령대: {target_age}, 성별: {target_gender}")
        
        safe_age = get_safe_label(le_age, target_age)
        safe_gender = get_safe_label(le_gender, target_gender)
        
        if str(target_age) != safe_age or str(target_gender) != safe_gender:
            print(f" 🤖 (AI 자동 보정) ➡️ 연령: '{safe_age}', 성별: '{safe_gender}'")
        
        age_code = le_age.transform([safe_age])[0]
        gender_code = le_gender.transform([safe_gender])[0]
        
        # 💡 [경고 해결] 예측 시 변수 이름을 포함한 DataFrame 형태로 전달
        input_df = pd.DataFrame([[age_code, gender_code]], columns=['연령대_코드', '성별_코드'])
        predicted_risk = rf_model.predict(input_df)[0]
        
        print(f" 🎯 AI 예측 리스크 점수: {predicted_risk:.1f}점")
        
        # 카탈로그 필터링 (다중 안전망 적용)
        age_prefix = str(target_age)[0]
        gender_kor = '남' if str(target_gender).upper() in ['M', '남', '남성'] else '여'
        
        # 1차 필터링 (나이대 + 성별)
        filtered_catalog = df_catalog[
            (df_catalog['가입나이'].astype(str).str.startswith(age_prefix)) & 
            (df_catalog['성별'] == gender_kor)
        ].copy()
        
        # 💡 [에러 해결] 2차 필터링 (조건에 맞는 데이터가 없으면 성별만 매칭)
        if filtered_catalog.empty:
            filtered_catalog = df_catalog[df_catalog['성별'] == gender_kor].copy()
        
        # 💡 [에러 해결] 3차 최후 필터링 (그래도 없으면 전체 데이터 사용)
        if filtered_catalog.empty:
            filtered_catalog = df_catalog.copy()
            
        if predicted_risk >= 50.0:
            print(" ⚠️ [추천 전략] 리스크가 높은 타겟입니다. '프리미엄 보장형' 위주로 추천합니다.")
            top_recs = filtered_catalog.sort_values(by='보장_충실도_점수', ascending=False).head(3)
        else:
            print(" 🛡️ [추천 전략] 리스크가 비교적 낮습니다. '가성비 실속형' 위주로 추천합니다.")
            top_recs = filtered_catalog.sort_values(by='가성비_점수', ascending=False).head(3)
            
        # 💡 [요청사항 반영] 어떤 회사에 어떤 보험인지 명확히 출력
        display_cols = ['추천_회사명', '상품플랜', '최종보험료', '보장_충실도_점수', '가성비_점수']
        result_df = top_recs[display_cols].reset_index(drop=True)
        
        # 💡 [에러 해결] 도출된 결과 개수만큼만 순위(Index)를 부여
        num_results = len(result_df)
        result_df.index = [f'{i+1}순위' for i in range(num_results)]
        
        result_df['보장_충실도_점수'] = result_df['보장_충실도_점수'].round(1)
        result_df['가성비_점수'] = result_df['가성비_점수'].round(1)
        
        print("-" * 80)
        print(result_df.to_string())
        print("-" * 80)
        
        return top_recs

    # ==========================================
    # 5. 라이브 테스트 실행
    # ==========================================
    print("\n🚀 [추천 엔진 라이브 테스트 시작]")
    
    result_60F = recommend_insurance('60대', 'F')
    result_20M = recommend_insurance('20대', 'M')
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        if not result_60F.empty:
            result_60F.to_excel(writer, sheet_name='60대_여성_추천', index=False)
        if not result_20M.empty:
            result_20M.to_excel(writer, sheet_name='20대_남성_추천', index=False)
        
    print(f"\n🎉 성공! 추천 결과가 저장되었습니다:\n 👉 {output_file}")

except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 1. 추천 엔진 초기화 중...
📊 2. 보험 상품별 가치 계산 및 보험사 정보 추출 중...

🚀 [추천 엔진 라이브 테스트 시작]

👤 [입력 프로필] 연령대: 60대, 성별: F
 🤖 (AI 자동 보정) ➡️ 연령: '60-69세', 성별: 'F'
 🎯 AI 예측 리스크 점수: 222.7점
 ⚠️ [추천 전략] 리스크가 높은 타겟입니다. '프리미엄 보장형' 위주로 추천합니다.
--------------------------------------------------------------------------------
    추천_회사명     상품플랜  최종보험료  보장_충실도_점수  가성비_점수
1순위     DB  DB_국내여행   5000     2631.9  5262.7
2순위     DB  DB_국내여행   2900     2631.9  9072.3
3순위     DB  DB_국내여행   8630     2631.9  3049.3
--------------------------------------------------------------------------------

👤 [입력 프로필] 연령대: 20대, 성별: M
 🤖 (AI 자동 보정) ➡️ 연령: '20-29세', 성별: 'M'
 🎯 AI 예측 리스크 점수: 3.0점
 🛡️ [추천 전략] 리스크가 비교적 낮습니다. '가성비 실속형' 위주로 추천합니다.
--------------------------------------------------------------------------------
    추천_회사명     상품플랜  최종보험료  보장_충실도_점수  가성비_점수
1순위     DB  DB_국내여행   2900     2631.9  9072.3
2순위     DB  DB_국내여행   5000     2631.9  5262.7
3순위     DB  DB_국내여행   8630     2631.9  3049.3
-----------------------------------